In [0]:
storage_account = "adlsgen2dreampod"
client_id = dbutils.secrets.get(scope="akv-scope",key="client-id")
client_secret = dbutils.secrets.get(scope="akv-scope", key="client-secret")
tenant_id = dbutils.secrets.get(scope="akv-scope", key="tenant-id")
spark.conf.set(f"fs.azure.account.oauth.type.{storage_account}.dfs.core.windows.net", "OAuth")
spark.conf.set(f"fs.azure.account.oauth.provider.type.{storage_account}.dfs.core.windows.net", 
              "org.apache.hadoop.bfs.oauth2.ClientCredentialProvider")
spark.conf.set(f"fs.azure.account.client.id.{storage_account}.dfs.core.windows.net", client_id)
spark.conf.set(f"fs.azure.account.client.secret.{storage_account}.dfs.core.windows.net", client_secret)
spark.conf.set(f"fs.azure.account.client.endpoint.{storage_account}.dfs.core.windows.net",
               f"https://login.microsoftonline.com/{tenant_id}/oauth2/token)

df_silver_path = f"abfss://silver@{storage_account}.dfs.core.windows.net"
df_silver_trips = spark.read.format("delta").load(df_silver_path)
df_silver_trips.createOrReplaceTemView("silver_trips")

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS gold;
use gold;

In [0]:
%sql
CREATE TABLE IF NOT EXISTS fact_trips (
    trip_id INT,
    driver_sk BIGINT,
    customer_sk BIGINT,
    pickup_location_sk BIGINT,
    dropoff_location_sk BIGINT,
    trip_start_time TIMESTAMP,
    trip_end_time TIMESTAMP,
    distance_km DECIMAL(18,2),
    fare_amount DECIMAL(18,2),
    payment_method STRING,
    trip_status STRING,
    _processed_at TIMESTAMP
) USING DELTA
LOCATION 'abfss://gold@adlsgen2dreampod.dfs.core.windows.net/dream_app/fact_trips';

CREATE OR REPLACE TEMPORARY VIEW v_fact_trips AS
SELECT
    t.trip_id,
    d.driver_sk,
    c.customer_sk,
    pl.location_sk AS pickup_location_sk ,
    dl.location_sk AS dropoff_location_sk,
    t.trip_start_timestamp AS trip_start_time,
    t.trip_end_timestamp AS trip_end_time,
    t.distance_km,
    t.fare_amount,
    t.payment_method,
    t.trip_status,
    current_timestamp() AS _processed_at
FROM
    silver_trips t
LEFT JOIN dim_drivers d ON t.driver_id = d.driver_id
    AND t.trip_start_time BETWEEN d.start_date AND nvl(d.end_date, '9999-12-31')
LEFT JOIN dim_customers c ON t.customer_id = c.customer_id
    AND t.trip_start_time BETWEEN c.start_date AND nvl(c.end_date, '9999-12-31')
LEFT JOIN dim_locations pl ON t.pickup_location = pl.location_id
LEFT JOIN dim_locations dl ON t.dropoff_location = dl.location_id;

SELECT * FROM v_fact_trips limit 10;
TRUNCATE TABLE fact_trips;
INSERT INTO fact_trips SELECT * FROM v_fact_trips;
OPTIMIZE fact_trips ZORDER BY (trip_start_time);